# Experiment 11 - XGBoost min_child_weight

This experiment keeps the current best XGBoost setup and changes only `min_child_weight` from 1 to 2.

The current best validation ROC-AUC is **0.941731**. This experiment is validation only.

In [1]:
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier


# Find the project root
current = Path.cwd().resolve()
project_root = None

for folder in [current, *current.parents]:
    if folder.name == "DataCompetition" and (folder / "data" / "train.csv").exists():
        project_root = folder
        break

if project_root is None:
    raise FileNotFoundError(
        "Could not find the DataCompetition project folder containing data/train.csv."
    )

train_path = project_root / "data" / "train.csv"
train = pd.read_csv(train_path)

print(f"Project root: {project_root}")
print(f"Training data: {train_path}")

# Same setup as the current best XGBoost experiment
X = train.drop(columns=["Will_Buy_EV", "id"])
y = train["Will_Buy_EV"].map({"No": 0, "Yes": 1})

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

X_train_processed = preprocessor.fit_transform(X_train)
X_valid_processed = preprocessor.transform(X_valid)

# Current best configuration.
# Only min_child_weight is changed from 1 to 2.
model = XGBClassifier(
    n_estimators=800,
    max_depth=5,
    learning_rate=0.04,
    min_child_weight=2,
    subsample=0.85,
    colsample_bytree=0.85,
    gamma=0,
    reg_alpha=0,
    reg_lambda=1,
    objective="binary:logistic",
    eval_metric="auc",
    tree_method="hist",
    random_state=42,
    n_jobs=-1
)

model.fit(X_train_processed, y_train)

valid_predictions = model.predict_proba(X_valid_processed)[:, 1]
roc_auc = roc_auc_score(y_valid, valid_predictions)

previous_best = 0.941731
difference = roc_auc - previous_best

print("")
print("============================================================")
print("                    EXPERIMENT RESULT")
print("============================================================")
print(f"Experiment 11 ROC-AUC : {roc_auc:.6f}")
print(f"Previous best         : {previous_best:.6f}")
print(f"Difference            : {difference:+.6f}")
print("")

if roc_auc > previous_best:
    print("NEW BEST MODEL")
else:
    print("Did not beat the current best.")

print("")
print("Configuration:")
print("n_estimators       = 800")
print("max_depth          = 5")
print("learning_rate      = 0.04")
print("min_child_weight   = 2  <- changed")
print("subsample          = 0.85")
print("colsample_bytree   = 0.85")
print("gamma              = 0")
print("reg_alpha          = 0")
print("reg_lambda         = 1")
print("")
print("Validation only. No submission was created.")

Project root: C:\Users\aakif\Documents\DataCompetition
Training data: C:\Users\aakif\Documents\DataCompetition\data\train.csv


C:\Users\aakif\AppData\Local\Temp\ipykernel_9304\4289371468.py:46: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X.select_dtypes(include=["object"]).columns.tolist()



                    EXPERIMENT RESULT
Experiment 11 ROC-AUC : 0.941747
Previous best         : 0.941731
Difference            : +0.000016

NEW BEST MODEL

Configuration:
n_estimators       = 800
max_depth          = 5
learning_rate      = 0.04
min_child_weight   = 2  <- changed
subsample          = 0.85
colsample_bytree   = 0.85
gamma              = 0
reg_alpha          = 0
reg_lambda         = 1

Validation only. No submission was created.
